# Quarto on Colab テンプレート

このノートブックは **Google Colab 上で Quarto を使いまくる完全ガイド**（quarto-plus）に対応したテンプレートです。
上のセルから順に実行すると、`.qmd` / `.ipynb` の作成・レンダリングを Colab 上で体験できます。

- ガイド: https://watanabe3tipapa.github.io/quarto-plus/docs/reference/colab-guide.html
- リポジトリ: https://github.com/watanabe3tipapa/quarto-plus

## Step 0: Quarto CLI をインストール

Colab には Quarto CLI が標準で入っていないため、このセルで Linux 用バイナリをダウンロード・展開します。
実行後はセッション内で `quarto` コマンドが使えるようになります。

In [ ]:
import os

QUARTO_VERSION = "1.6.43"  # 必要に応じて最新版に変更
INSTALL_DIR = "/usr/local/quarto"
DEB_URL = f"https://github.com/quarto-dev/quarto-cli/releases/download/v{QUARTO_VERSION}/quarto-{QUARTO_VERSION}-linux-amd64.tar.gz"

!mkdir -p /tmp/quarto-install
!wget -q -O /tmp/quarto-install/quarto.tar.gz {DEB_URL}
!mkdir -p {INSTALL_DIR}
!tar -xzf /tmp/quarto-install/quarto.tar.gz -C {INSTALL_DIR} --strip-components=1

os.environ["PATH"] = f"{INSTALL_DIR}/bin:{os.environ.get('PATH','')}"

!quarto --version

## Step 1: `.qmd` を作成して HTML にレンダリング

次のセルでサンプルの `.qmd`（実行コード付き）を作成し、`quarto render --to html` で HTML に変換します。

In [ ]:
from pathlib import Path

qmd_source = '''---
title: "Colab + Quarto Demo"
author: "Your Name"
format:
  html:
    toc: true
    code-fold: true
    theme: cosmo
jupyter: python3
---

## データの可視化

```python
#| label: fig-demo
#| fig-cap: "サンプルプロット"
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)
y = np.sin(x)

plt.figure(figsize=(8, 4))
plt.plot(x, y, color="steelblue", linewidth=2)
plt.title("Sine Wave")
plt.show()
```

## まとめ

Colab 上で Quarto を使えば、コード・出力・解説を一括管理できます。
'''

Path("/content/report.qmd").write_text(qmd_source, encoding="utf-8")
print("saved: /content/report.qmd")

In [ ]:
!quarto render /content/report.qmd --to html
!ls -la /content/report*

## Step 2: `.ipynb` を Quarto でレンダリング

Colab で作ったノートブックも Quarto でドキュメント化できます。アップロードした ipynb をこのセルで指定してください。

> Colab でダウンロードした ipynb は kernelspec の `language` キーが欠落していることがあります。次のセルが自動で補完します。

In [ ]:
import json
from pathlib import Path

# アップロードしたノートブックをここで指定
nb_path = Path("/content/yournote.ipynb")

if nb_path.exists():
    nb = json.loads(nb_path.read_text(encoding="utf-8"))
    ks = nb.setdefault("metadata", {}).setdefault("kernelspec", {})
    ks.setdefault("language", "python")
    nb_path.write_text(json.dumps(nb, indent=2, ensure_ascii=False), encoding="utf-8")
    !quarto render {nb_path} --to html --execute
else:
    print("notebook not found. upload the file and set nb_path.")

## Step 3: Quarto プロジェクトを動かす

`_quarto.yml` を持つ website / book プロジェクトも Colab 上でビルドできます。

In [ ]:
from pathlib import Path

!mkdir -p /content/myquarto
%cd /content/myquarto

quarto_yml = '''project:
  type: website
  output-dir: _site

website:
  title: "My Quarto Site"
  navbar:
    left:
      - href: index.qmd
        text: Home

format:
  html:
    theme: cosmo
    toc: true
'''
Path("_quarto.yml").write_text(quarto_yml, encoding="utf-8")

index_qmd = '''---
title: "Home"
---

## ようこそ

quarto-plus のテンプレートで Quarto プロジェクトを始められます。
'''
Path("index.qmd").write_text(index_qmd, encoding="utf-8")

!quarto render
!ls -R _site

## Step 4: Google Drive と連携して永続化

Colab のセッションは破棄されるため、作業ディレクトリを Google Drive に置くと再利用できます。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
work_dir = "/content/drive/MyDrive/quarto-work"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

!quarto --version

## コマンド早見表

| 目的 | コマンド |
|---|---|
| qmd → HTML | `quarto render file.qmd --to html` |
| qmd → PDF | `quarto render file.qmd --to pdf` |
| qmd → docx | `quarto render file.qmd --to docx` |
| ipynb → HTML（実行あり） | `quarto render file.ipynb --to html --execute` |
| プロジェクト全体 | `quarto render`（_quarto.yml のある場所で） |
| ipynb ↔ qmd 変換 | `quarto convert file.ipynb` / `quarto convert file.qmd` |

## トラブルシューティング

- `quarto: command not found` → Step 0 の PATH 設定がセッション内で消えています。`os.environ["PATH"]` に `/usr/local/quarto/bin` を再追加してください。
- `language key missing` → Step 2 のセルが kernelspec の `language` を自動補完します。
- PDF が出ない → `!quarto install tool tinytex` で TeX 環境（TinyTeX）を導入してください。

## 実践用例

e-Stat データの分析レポート / ブログ公開 / ダッシュボード / OKF プロジェクトなど、より実用的な用例はガイドの「実践用例」節にまとめています。

https://watanabe3tipapa.github.io/quarto-plus/docs/reference/colab-guide.html

## 注意点

- Colab のセッションは約 12 時間で切れ、ランタイムの状態は消えます。作業成果は Google Drive へ保存してください。
- Quarto の詳細は公式ドキュメント https://quarto.org を参照してください。
- このテンプレートに対応する実践ガイドは https://watanabe3tipapa.github.io/quarto-plus/docs/reference/colab-guide.html を参照してください。